# CrisisBench
The Crisis Benchmarks Dataset consists of data from different data sources [1]):

## A. Original description

### Data format and directories

The data directory contains the following sub-directories. In each directory, there are files for Informativeness and Humanitarian tasks.

* data/all_data_en -- all combined english dataset used for the experiments
* data/individual_data_en -- consists of data used for the experiments as individual data source such as crisisnlp and crisislex
* data/event_aware_en -- all combined english dataset with event tag (fire, earthquake, flood, ...) are tagged
* data/data_split_all_lang -- all combined dataset with their train/dev and test splits
* data/initial_filtering -- all combined dataset duplicate removed data
* data/class_label_mapped -- all combined dataset initial set of dataset where class label mapped

#### Format of the TSV files

Each TSV file contains the following columns, separated by a tab:

* id: corresponds to the user tweet id from Twitter.
* event: event name associated with the respective dataset
* source: source of the dataset
* text: tweet text.
* lang: language tag obtained either from Twitter or obtained from Google language detection API
* lang conf: confidence score obtained from Google language detection API; for many cases, there are tag "NA", which represents that the language tag is obtained from Twitter from Google API
* class_label: class label a given tweet text.


### Citation

* Firoj Alam, Hassan Sajjad, Muhammad Imran and Ferda Ofli, CrisisBench: Benchmarking Crisis-related Social Media Datasets for Humanitarian Information Processing, In ICWSM, 2021.
* Muhammad Imran, Prasenjit Mitra, Carlos Castillo. Twitter as a Lifeline: Human-annotated Twitter Corpora for NLP of Crisis-related Messages. In Proceedings of the 10th Language Resources and Evaluation Conference (LREC), 2016, Slovenia.
* A. Olteanu, S. Vieweg, C. Castillo. 2015. What to Expect When the Unexpected Happens: Social Media Communications Across Crises. In Proceedings of the ACM 2015 Conference on Computer Supported Cooperative Work and Social Computing (CSCW '15). ACM, Vancouver, BC, Canada.
* A. Olteanu, C. Castillo, F. Diaz, S. Vieweg. 2014. CrisisLex: A Lexicon for Collecting and Filtering Microblogged Communications in Crises. In Proceedings of the AAAI Conference on Weblogs and Social Media (ICWSM'14). AAAI Press, Ann Arbor, MI, USA.
* Muhammad Imran, Shady Elbassuoni, Carlos Castillo, Fernando Diaz and Patrick Meier. Practical Extraction of Disaster-Relevant Information from Social Media. In Social Web for Disaster Management (SWDM'13) - Co-located with WWW, May 2013, Rio de Janeiro, Brazil.
* Muhammad Imran, Shady Elbassuoni, Carlos Castillo, Fernando Diaz and Patrick Meier. Extracting Information Nuggets from Disaster-Related Messages in Social Media. In Proceedings of the 10th International Conference on Information Systems for Crisis Response and Management (ISCRAM), May 2013, Baden-Baden, Germany.

## B. Overview

In [1]:
from pathlib import Path
import os
import csv
import pandas as pd

import configuration
from src import dataset_settings

from dotenv import load_dotenv
load_dotenv()

dataset_path = Path(os.getenv("DATASETS_PATH")) / "CrisisBench" / "data"

There are three english-related sub sets, which author did not clearly explain.

1. class_label_mapped

    Overlapped but not identical.
    It is the Second column `Mapping` on Table 1 of the paper
    
    - crisis_consolidated_humanitarian.tsv
    - crisis_consolidated_informativeness.tsv

1. initial_filtering

    The `*_filtered_lang` have language detection columns, while the `*_filtered` have `text_formatted` column. They are identical set of tweets

    `humanitarian` and `informativeness` are overlapped but not indentical

    It is the Third column `Filtering` on Table 1 of the paper

    - crisis_consolidated_humanitarian_filtered_lang.tsv
    - crisis_consolidated_humanitarian_filtered.tsv
    - crisis_consolidated_informativeness_filtered_lang.tsv
    - crisis_consolidated_informativeness_filtered.tsv

1. all_data_en

    Overlapped but not identical

    - crisis_consolidated_humanitarian_filtered_lang_en_dev.tsv
    - crisis_consolidated_humanitarian_filtered_lang_en_test.tsv
    - crisis_consolidated_humanitarian_filtered_lang_en_train.tsv
    - crisis_consolidated_informativeness_filtered_lang_en_dev.tsv
    - crisis_consolidated_informativeness_filtered_lang_en_test.tsv
    - crisis_consolidated_informativeness_filtered_lang_en_train.tsv

According to the paper, the order of sets is `class_label_mapped` > `initial_filtering` > `all_data_en`.

Then
- `class_label_mapped` should be the supper set of both `initial_filtering` and `all_data_en`
- `initial_filtering` should be the supper set of `all_data_en`

However, the reality is,
- only `class_label_mapped` is the supper set of `initial_filtering`.
- `all_data_en` contains 443 record from `drd-figureeight-multimedia` and 4 records from `aidr_system` that are not existed in `class_label_mapped` and `initial_filtering`.

=> TODO:
- Merge 443 and 4 records from `all_data_en` back to `class_label_mapped`
- Select English only

**all_data_en**

In [14]:
base_dir = dataset_path / "all_data_en"

humanitarian_files = sorted(
    base_dir.rglob("crisis_consolidated_humanitarian_filtered_lang_en_*.tsv")
)

informativeness_files = sorted(
    base_dir.rglob("crisis_consolidated_informativeness_filtered_lang_en_*.tsv")
)

df_hum_all_data_en_ids = set(pd.concat(
    [pd.read_csv(f, sep="\t") for f in humanitarian_files], ignore_index=True
)['id'])

df_inf_all_data_en = pd.concat(
    [
        pd.read_csv(
            f,
            sep="\t",
            quoting=csv.QUOTE_NONE,
            on_bad_lines="skip",
        )
        for f in informativeness_files
    ],
    ignore_index=True,
)

df_inf_all_data_en_ids = set(df_inf_all_data_en['id'])

In [3]:
print(len(df_inf_all_data_en_ids))
print(len(df_hum_all_data_en_ids))
print(len(df_inf_all_data_en_ids - df_hum_all_data_en_ids))

156896
87454
69488


**class_label_mapped**

In [22]:
base_dir = dataset_path / "class_label_mapped"

humanitarian_file =  base_dir / "crisis_consolidated_humanitarian.tsv"
informativeness_file = base_dir / "crisis_consolidated_informativeness.tsv"

df_hum_class_label_mapped_ids = set(pd.read_csv(humanitarian_file, sep="\t")["id"])
df_inf_class_label_mapped = pd.read_csv(informativeness_file, sep="\t")
df_inf_class_label_mapped_ids = set(df_inf_class_label_mapped["id"])

In [23]:
df_inf_class_label_mapped.head()

,id,event,source,text,class_label
0,368659239272579073,2013_Manila_Floods,CrisisLexT26,RT @dost_pagasa: WEATHER BULLETIN No. 2 Tropic...,informative
1,368673453785636864,2013_Manila_Floods,CrisisLexT26,"RT @24_Oras: Sa datos ng @dost_pagasa, huling ...",informative
2,368676146541371392,2013_Manila_Floods,CrisisLexT26,Fuck ayoko ng baha. Paano ako sa Manila gdi pu...,not_informative
3,368679682322874371,2013_Manila_Floods,CrisisLexT26,RT @ANCALERTS: Tropical Depression Maring is m...,informative
4,368929037877403652,2013_Manila_Floods,CrisisLexT26,Intramuros baha. I miss Manila. http://t.co/Uk...,informative


In [24]:
df_inf_class_label_mapped.value_counts("source")

source
CrisisLexT6                   59896
CrisisNLP-volunteers          26833
CrisisLexT26                  24511
CrisisNLP-CF                  24438
DRD-FigureEight-Multimedia    21074
CrisisMMD                     16058
DSM-CF                        10800
AIDR_system                    7392
ISCRAM13                       3196
SWDM13                         1344
Name: count, dtype: int64

In [5]:
print(len(df_hum_class_label_mapped_ids))
print(len(df_inf_class_label_mapped_ids))

print(len(df_inf_class_label_mapped_ids - df_hum_class_label_mapped_ids))

167805
195473
27803


**initial_filtering**

In [20]:
base_dir = dataset_path / "initial_filtering"

humanitarian = base_dir / 'crisis_consolidated_humanitarian_filtered_lang.tsv'
humanitarian_lang = base_dir / 'crisis_consolidated_humanitarian_filtered.tsv'
informativeness = base_dir / 'crisis_consolidated_informativeness_filtered_lang.tsv'
informativeness_lang = base_dir / 'crisis_consolidated_informativeness_filtered.tsv'

# df_hum_init = pd.read_csv(humanitarian, sep="\t")
# df_hum_init_lang = pd.read_csv(humanitarian_lang, sep="\t")
# df_inf_init = pd.read_csv(informativeness, sep="\t")
# df_inf_init_lang = pd.read_csv(informativeness_lang, sep="\t")

df_hum_init_ids = set(pd.read_csv(humanitarian, sep="\t")["id"])
df_hum_init_lang_ids = set(pd.read_csv(humanitarian_lang, sep="\t")["id"])
df_inf_init_ids = set(pd.read_csv(informativeness, sep="\t")["id"])
df_inf_init_lang = pd.read_csv(informativeness_lang, sep="\t")
df_inf_init_lang_ids = set(df_inf_init_lang["id"])

In [25]:
df_inf_init_lang.head()

,id,event,source,text_original,text_formatted,class_label
0,368659239272579073,2013_manila_floods,crisislext26,RT @dost_pagasa: WEATHER BULLETIN No. 2 Tropic...,rt weather bulletin no tropical cyclone alert ...,informative
1,368673453785636864,2013_manila_floods,crisislext26,"RT @24_Oras: Sa datos ng @dost_pagasa, huling ...",rt sa datos ng huling namataan ang td maringph...,informative
2,368676146541371392,2013_manila_floods,crisislext26,Fuck ayoko ng baha. Paano ako sa Manila gdi pu...,fuck ayoko ng baha paano ako sa manila gdi pur...,not_informative
3,368679682322874371,2013_manila_floods,crisislext26,RT @ANCALERTS: Tropical Depression Maring is m...,rt tropical depression maring is moving east s...,informative
4,368929037877403652,2013_manila_floods,crisislext26,Intramuros baha. I miss Manila. http://t.co/Uk...,intramuros baha i miss manila url,informative


In [26]:
df_inf_init_lang.value_counts("source")

source
crisislext6                   49806
crisisnlp-volunteers          22013
drd-figureeight-multimedia    20452
crisislext26                  19893
crisisnlp-cf                  18388
crisismmd                     16020
dsm-cf                         8835
aidr_system                    6865
iscram13                       2521
swdm13                          857
Name: count, dtype: int64

In [31]:
# crisislext6                   49806
# crisislext26                  19893
print(49806 + 19893)
# crisisnlp-volunteers          22013
# crisisnlp-cf                  18388
print(22013 + 18388)
# swdm13                          857
# iscram13                       2521
# drd-figureeight-multimedia    20452
# dsm-cf                         8835
# crisismmd                     16020
# aidr_system                    6865

# source
# drd-figureeight-multimedia    443
# aidr_system                     4

print(20452+443)

69699
40401
20895


In [7]:
print(len(df_hum_init_ids))
print(len(df_hum_init_lang_ids))
print(len(df_inf_init_ids))
print(len(df_inf_init_lang_ids))

print(len(df_hum_init_ids - df_hum_init_lang_ids))
print(len(df_inf_init_ids - df_inf_init_lang_ids))
print(len(df_hum_init_lang_ids - df_hum_init_ids))
print(len(df_inf_init_lang_ids - df_inf_init_ids))

141455
141455
165647
165647
0
0
0
0


# Check which set is parent set

In [10]:
print(len(df_inf_all_data_en_ids))
print(len(df_inf_init_ids))
print(len(df_inf_class_label_mapped_ids))

print(df_inf_init_ids.issuperset(df_inf_all_data_en_ids))
print(df_inf_class_label_mapped_ids.issuperset(df_inf_all_data_en_ids))
print(df_inf_class_label_mapped_ids.issuperset(df_inf_init_ids))

# print(len(df_inf_all_data_en_ids - df_inf_init_ids))
# print(len(df_inf_init_ids - df_inf_class_label_mapped_ids))
# print(len(df_inf_all_data_en_ids - df_inf_class_label_mapped_ids))

156896
165647
195473
False
False
True


In [12]:
print(len(df_inf_all_data_en_ids - df_inf_init_ids))
print(len(df_inf_all_data_en_ids - df_inf_class_label_mapped_ids))

(df_inf_all_data_en_ids - df_inf_init_ids) - df_inf_all_data_en_ids - df_inf_class_label_mapped_ids

447
447


set()

In [18]:
diff_df = df_inf_all_data_en[df_inf_all_data_en['id'].isin(df_inf_all_data_en_ids - df_inf_init_ids)]
diff_df.head()

,id,event,source,text,lang,lang_conf,class_label
35,24960,disaster_events,drd-figureeight-multimedia,Additional projects are underway using the 'bu...,en,1.0,informative
532,19296,disaster_events,drd-figureeight-multimedia,The Punjab is currently below normal for cumul...,en,1.0,informative
760,24943,disaster_events,drd-figureeight-multimedia,Since this morning some of affected community ...,en,1.0,informative
1165,24940,disaster_events,drd-figureeight-multimedia,The official with the Gansu Provincial Civil A...,en,1.0,informative
1279,18743,disaster_events,drd-figureeight-multimedia,The flooding has been caused by persistent hea...,en,1.0,informative


In [30]:
diff_df.value_counts("source")

source
drd-figureeight-multimedia    443
aidr_system                     4
Name: count, dtype: int64

In [21]:
df_inf_init_lang.value_counts("event")

event
disaster_events                    36152
2015_nepal_earthquake              11032
2012_sandy_hurricane-ontopic        8919
2013_oklahoma_tornado-ontopic       8824
2013_alberta_floods-ontopic         8753
                                   ...  
2015_vanuatu_cyclone-pam             502
iraq_iran_earthquake                 493
2014_iceland_volcano                 294
2012_us_sandy-hurricane-a144267      283
2014_malaysia_airline                121
Name: count, Length: 61, dtype: int64

In [19]:
diff_df.value_counts("source")

source
drd-figureeight-multimedia    443
aidr_system                     4
Name: count, dtype: int64

In [ ]:
# all_files = sorted(base_dir.rglob('**/*.tsv'))

# dfs = []
# for file in all_files:
#     parent_name = file.parent.name
#     if parent_name not in events:
#         print(f'Skipping: {parent_name}')
#         continue
#     data = events[parent_name]
#     df = pd.read_csv(file, sep='\t', encoding='utf_8')
#     df['tweet_id'] = df['tweet_id'].astype('int64')
#     df['event_type'] = data['event_type']
#     df['event_name'] = data['event_name']
#     df['year'] = data['year']
#     df['dataset'] = 'HumAID'
#     df['meta'] = df.apply(
#         lambda x: {
#             'file_name': file.name,
#             'sub_dataset': 'Crowdflower'
#             },
#         axis=1)
#     dfs.append(df)

# df = pd.concat(dfs, ignore_index=True)